# DeepRacer 赛道线优化工作流

## 项目简介

本 Notebook 整合 DeepRacer 赛道优化的完整流程，适合学生学习使用。

### 工作流程

1. **Step 1**: 将赛道 npy 文件导出为 waypoints.csv
2. **Step 2**: 计算最优赛车线（带安全距离）
3. **Step 3**: 计算速度剖面、到达时间和动作空间
4. **Step 4**: 导出四维赛车线数据（用于可视化）
5. **Step 5**: 奖励函数

---

## 可调节参数

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `npy_path` | `"./map/reinvent_base.npy"` | 赛道文件路径 |
| `safety_distance` | `0.2` | 赛车线与边界最小安全距离 (米) |
| `line_iterations` | `2000` | 赛车线优化迭代次数 |
| `xi_iterations` | `4` | 每点二分查找深度 |
| `min_speed` | `1.0` | 最小速度 (m/s) |
| `max_speed` | `3.0` | 最大速度 (m/s) |
| `look_ahead_points` | `1` | 前视点数（提前预判弯道） |
| `n_clusters` | `21` | Action Space 动作数量 |

## Step 1: 环境配置

In [1]:
# 导入核心函数（来自 deepracer-notebook/core.py）
from core import compute_racing_line, npy_to_csv

print("✅ 核心函数加载完成（独立模式）")

✅ 核心函数加载完成（独立模式）


---

## Step 2: 赛道数据导入 → CSV

In [2]:
from core import npy_to_csv

# ============== 可调节参数 ==============
npy_path = "./map/reinvent_base.npy"  # 赛道 npy 文件路径
csv_output_path = "./waypoints.csv"    # 导出 CSV 路径
# ======================================

# 执行转换
npy_to_csv(npy_path, csv_output_path)
print(f"\n✅ waypoints.csv 已生成: {csv_output_path}")
print("提示: 此文件可在 visualize.html 中导入查看赛道边界")

✅ 已导出为 ./waypoints.csv

✅ waypoints.csv 已生成: ./waypoints.csv
提示: 此文件可在 visualize.html 中导入查看赛道边界


## Step 3: 计算赛车线

使用 `core.py` 中的 `compute_racing_line()` 函数，在保证安全距离的前提下优化赛车线。

In [3]:
from core import compute_racing_line
import numpy as np

# ============== 可调节参数 ==============
npy_path = "./map/reinvent_base.npy"
safety_distance = 0.2   # 与赛道边界的最小安全距离 (米)
line_iterations = 2000  # 优化迭代次数
xi_iterations = 4       # 二分查找深度
# ======================================

print("正在计算赛车线，请稍候...")
print(f"  安全距离: {safety_distance} m")
print(f"  迭代次数: {line_iterations}")

racing_line, waypoints = compute_racing_line(
    track_path=npy_path,
    safety_distance=safety_distance,
    line_iterations=line_iterations,
    xi_iterations=xi_iterations,
    verbose=True
)

print(f"\n✅ 赛车线计算完成！共 {len(racing_line)} 个点")

正在计算赛车线，请稍候...
  安全距离: 0.2 m
  迭代次数: 2000
加载赛道: ./map/reinvent_base.npy
赛道点数: 119, 安全距离: 0.2
开始优化赛车线...
  迭代 0/2000
  迭代 200/2000
  迭代 400/2000
  迭代 600/2000
  迭代 800/2000
  迭代 1000/2000
  迭代 1200/2000
  迭代 1400/2000
  迭代 1600/2000
  迭代 1800/2000
原中心线长度: 17.71
优化后赛车线长度: 16.52

✅ 赛车线计算完成！共 119 个点


### 赛车线坐标预览

In [4]:
# 去除闭合点（首尾重复），生成纯坐标列表
if len(racing_line) > 1 and np.allclose(racing_line[0], racing_line[-1]):
    racing_line_pts = racing_line[:-1]
else:
    racing_line_pts = racing_line

print("赛车线坐标 (共 {} 点):".format(len(racing_line_pts)))
print("RACING_LINE = [")
for pt in racing_line_pts:
    print(f"    [{pt[0]}, {pt[1]}],")
print("]")

赛车线坐标 (共 118 点):
RACING_LINE = [
    [3.0598383880879805, 0.6824625084934601],
    [3.2094861758689115, 0.6825019585549194],
    [3.359260476617272, 0.6829505288164354],
    [3.509030084284507, 0.6832698762789144],
    [3.6587949991226196, 0.6834610402584076],
    [3.808555006980896, 0.6835170090198517],
    [3.9583150148391724, 0.6835691034793854],
    [4.1080756187438965, 0.6836211383342743],
    [4.2578349113464355, 0.6836741119623184],
    [4.407594919204712, 0.683727964758873],
    [4.5572767271940595, 0.6840228977793554],
    [4.706778609903321, 0.684862767541],
    [4.855990205861853, 0.686558014707021],
    [5.004794529676611, 0.6894188099462517],
    [5.15306158903889, 0.693770727479484],
    [5.300646866784055, 0.699954544930321],
    [5.447384833459565, 0.7083378471158852],
    [5.593064687822428, 0.7193674867121893],
    [5.737440474304398, 0.7335286952621685],
    [5.8802099608002525, 0.7513797156825898],
    [6.0209912905148855, 0.7735794215201868],
    [6.159386794606261

---

## Step 4: 计算速度剖面、到达时间和动作空间

根据赛车线计算每点的最优速度、转弯角度、到达时间，并生成 DeepRacer Action Space。

In [ ]:
import numpy as np
import math

# ============== 可调节参数 ==============
min_speed = 1.3          # 最小速度 (m/s)
max_speed = 2.6          # 最大速度 (m/s)
look_ahead_points = 1    # 前视点数（速度计算时向前看的点数）
dist_wheels_front_back = 0.165  # 前后轮距离（米，用于转向计算）
n_clusters = 21          # Action Space 动作数量
# ======================================

# 准备赛车线数据
track = racing_line_pts.copy()  # 使用去除闭合点的版本
print(f"赛车线点数: {len(track)}")

# ---------- 辅助函数 ----------

def circle_radius(coords):
    """通过三点计算转弯半径"""
    x1, y1, x2, y2, x3, y3 = [i for sub in coords for i in sub]
    a = x1*(y2-y3) - y1*(x2-x3) + x2*y3 - x3*y2
    b = (x1**2+y1**2)*(y3-y2) + (x2**2+y2**2)*(y1-y3) + (x3**2+y3**2)*(y2-y1)
    c = (x1**2+y1**2)*(x2-x3) + (x2**2+y2**2)*(x3-x1) + (x3**2+y3**2)*(x1-x2)
    d = (x1**2+y1**2)*(x3*y2-x2*y3) + (x2**2+y2**2)*(x1*y3-x3*y1) + (x3**2+y3**2)*(x2*y1-x1*y2)
    try:
        r = abs((b**2+c**2-4*a*d) / abs(4*a**2)) ** 0.5
    except:
        r = 999
    return r

def circle_indexes(mylist, index_car, add_index_1=0, add_index_2=0):
    """环形索引"""
    list_len = len(mylist)
    index_1 = (index_car + add_index_1) % list_len
    index_2 = (index_car + add_index_2) % list_len
    return [index_car, index_1, index_2]

def dist_2_points(x1, x2, y1, y2):
    """两点间距离"""
    return abs(abs(x1-x2)**2 + abs(y1-y2)**2)**0.5

def is_left_curve(coords):
    """判断弯道方向是否为左转"""
    x1, y1, x2, y2, x3, y3 = [i for sub in coords for i in sub]
    return ((x2-x1)*(y3-y1) - (y2-y1)*(x3-x1)) > 0

# ---------- 1. 计算速度剖面 ----------

radius = []
for i in range(len(track)):
    indexes = circle_indexes(track, i, add_index_1=-1, add_index_2=1)
    coords = [track[indexes[0]], track[indexes[1]], track[indexes[2]]]
    radius.append(circle_radius(coords))

v_min_r = min(radius)**0.5
constant_multiple = min_speed / v_min_r
print(f"速度常数倍数: {constant_multiple:.4f}")

# 前视模式速度计算
velocity = []
for i in range(len(track)):
    next_n_radius = []
    for j in range(look_ahead_points + 1):
        index = circle_indexes(radius, i, add_index_1=j)[1]
        next_n_radius.append(radius[index])
    r_min = min(next_n_radius)
    v = constant_multiple * r_min**0.5
    velocity.append(min(v, max_speed))

# ---------- 2. 计算距离和时间 ----------

distance_to_prev = []
for i in range(len(track)):
    indexes = circle_indexes(track, i, add_index_1=-1, add_index_2=0)[0:2]
    coords = [track[indexes[0]], track[indexes[1]]]
    dist_to_prev = dist_2_points(
        x1=coords[0][0], x2=coords[1][0],
        y1=coords[0][1], y2=coords[1][1]
    )
    distance_to_prev.append(dist_to_prev)

time_to_prev = [(distance_to_prev[i] / velocity[i]) for i in range(len(track))]
total_time = sum(time_to_prev)

print(f"理论最优圈速: {total_time:.2f} s")

### 四维赛车线数据（用于 visualize.html 可视化）

格式: `[[x, y, speed, time], ...]`

**操作**: 将下方输出复制到 `visualize.html` 的「赛车线」文本框中

In [6]:
# 构建四维赛车线数据
racing_track_everything = []
for i in range(len(track)):
    racing_track_everything.append([
        round(track[i][0], 6),
        round(track[i][1], 6),
        round(velocity[i], 4),
        round(time_to_prev[i], 4)
    ])

print("=" * 60)
print("四维赛车路径点 (复制以下内容到 visualize.html):")
print("=" * 60)
for pt in racing_track_everything:
    print(f"[{pt[0]}, {pt[1]}, {pt[2]}, {pt[3]}],")

四维赛车路径点 (复制以下内容到 visualize.html):
[3.059838, 0.682463, 2.6, 0.0574],
[3.209486, 0.682502, 2.6, 0.0576],
[3.35926, 0.682951, 2.6, 0.0576],
[3.50903, 0.68327, 2.6, 0.0576],
[3.658795, 0.683461, 2.6, 0.0576],
[3.808555, 0.683517, 2.6, 0.0576],
[3.958315, 0.683569, 2.6, 0.0576],
[4.108076, 0.683621, 2.6, 0.0576],
[4.257835, 0.683674, 2.6, 0.0576],
[4.407595, 0.683728, 2.6, 0.0576],
[4.557277, 0.684023, 2.6, 0.0576],
[4.706779, 0.684863, 2.6, 0.0575],
[4.85599, 0.686558, 2.6, 0.0574],
[5.004795, 0.689419, 2.6, 0.0572],
[5.153062, 0.693771, 2.6, 0.0571],
[5.300647, 0.699955, 2.6, 0.0568],
[5.447385, 0.708338, 2.6, 0.0565],
[5.593065, 0.719367, 2.6, 0.0562],
[5.73744, 0.733529, 2.6, 0.0558],
[5.88021, 0.75138, 2.6, 0.0553],
[6.020991, 0.773579, 2.6, 0.0548],
[6.159387, 0.80073, 2.3642, 0.0597],
[6.294655, 0.833986, 2.0784, 0.067],
[6.42574, 0.874811, 1.8218, 0.0754],
[6.551212, 0.924848, 1.7152, 0.0788],
[6.669016, 0.985982, 1.5008, 0.0884],
[6.777697, 1.058565, 1.3, 0.1005],
[6.873676, 1.144

## Step 5: 奖励函数

将以下奖励函数直接复制到 AWS DeepRacer 控制台的奖励函数编辑器中即可使用。

In [7]:
import math

# ======================================================================
# 1. 填入你的 4维 最优赛车线数据: [x, y, target_speed, time_to_prev]
# ======================================================================
OPTIMAL_RACING_LINE = [
    [3.06111, 0.68648, 3.0, 0.04943],
    [3.21004, 0.68428, 3.0, 0.04965],
    [3.3594, 0.68338, 3.0, 0.04979],
    [3.50903, 0.68327, 3.0, 0.04988],
    [3.65879, 0.68347, 3.0, 0.04992],
    [3.80844, 0.68382, 3.0, 0.04988],
    [3.95802, 0.68432, 3.0, 0.04986],
    [4.10752, 0.68504, 3.0, 0.04983],
    [4.25686, 0.68614, 3.0, 0.04978],
    [4.40599, 0.68779, 3.0, 0.04971],
    [4.55482, 0.69014, 3.0, 0.04962],
    [4.70328, 0.69338, 3.0, 0.0495],
    [4.85128, 0.69772, 3.0, 0.04935],
    [4.9987, 0.70338, 3.0, 0.04918],
    [5.14543, 0.71061, 3.0, 0.04897],
    [5.29132, 0.7197, 3.0, 0.04872],
    [5.43619, 0.73099, 3.0, 0.04843],
    [5.5798, 0.7449, 3.0, 0.04809],
    [5.72191, 0.76182, 3.0, 0.04771],
    [5.86227, 0.78216, 3.0, 0.04728],
    [6.00056, 0.80637, 2.71862, 0.05164],
    [6.13627, 0.83521, 2.364, 0.05869],
    [6.26874, 0.86956, 2.09075, 0.06545],
    [6.39688, 0.91081, 1.83817, 0.07324],
    [6.51941, 0.96032, 1.63194, 0.08098],
    [6.6345, 1.01966, 1.44548, 0.08958],
    [6.73986, 1.09028, 1.26612, 0.10017],
    [6.83221, 1.17352, 1.26612, 0.0982],
    [6.9064, 1.27019, 1.47988, 0.08234],
    [6.9674, 1.37418, 1.38224, 0.08722],
    [7.01402, 1.48444, 1.25, 0.09578],
    [7.04368, 1.59998, 1.25, 0.09543],
    [7.05197, 1.7186, 1.34269, 0.08856],
    [7.04168, 1.83677, 1.40194, 0.0846],
    [7.01458, 1.95222, 1.44616, 0.08201],
    [6.97206, 2.06331, 1.29518, 0.09184],
    [6.9153, 2.16874, 1.29518, 0.09245],
    [6.84048, 2.26418, 1.47155, 0.08241],
    [6.75235, 2.34995, 1.49161, 0.08245],
    [6.6517, 2.42474, 1.67407, 0.07491],
    [6.54137, 2.48961, 1.87034, 0.06843],
    [6.42337, 2.54568, 2.11113, 0.06189],
    [6.29934, 2.59425, 2.44656, 0.05445],
    [6.17077, 2.63685, 2.86154, 0.04733],
    [6.03885, 2.67483, 3.0, 0.04576],
    [5.9047, 2.70966, 3.0, 0.0462],
    [5.76943, 2.7429, 3.0, 0.04643],
    [5.63199, 2.77735, 3.0, 0.04723],
    [5.495, 2.81294, 3.0, 0.04718],
    [5.35867, 2.85022, 3.0, 0.04711],
    [5.22326, 2.88975, 3.0, 0.04702],
    [5.08905, 2.93214, 2.9908, 0.04706],
    [4.95646, 2.97829, 2.63367, 0.05331],
    [4.82601, 3.02922, 2.30421, 0.06078],
    [4.69836, 3.08626, 2.30421, 0.06067],
    [4.57447, 3.1512, 2.78758, 0.05018],
    [4.4532, 3.22155, 2.96185, 0.04733],
    [4.33429, 3.29671, 3.0, 0.04689],
    [4.2175, 3.37612, 3.0, 0.04708],
    [4.10257, 3.45923, 3.0, 0.04728],
    [3.98933, 3.54559, 3.0, 0.04747],
    [3.87772, 3.63473, 3.0, 0.04761],
    [3.76883, 3.7256, 3.0, 0.04727],
    [3.65881, 3.81366, 3.0, 0.04697],
    [3.5472, 3.89847, 3.0, 0.04673],
    [3.43373, 3.97944, 2.76037, 0.0505],
    [3.31809, 4.05587, 2.47663, 0.05597],
    [3.19992, 4.12686, 2.17043, 0.06352],
    [3.07872, 4.19114, 1.92107, 0.07141],
    [2.9538, 4.24671, 1.92107, 0.07117],
    [2.82466, 4.29102, 2.13805, 0.06386],
    [2.6927, 4.32605, 2.25905, 0.06044],
    [2.55872, 4.35264, 2.34372, 0.05828],
    [2.42335, 4.37127, 2.38028, 0.05741],
    [2.28712, 4.38211, 2.40961, 0.05671],
    [2.15058, 4.38532, 2.34595, 0.05822],
    [2.01429, 4.38102, 2.25866, 0.06037],
    [1.8789, 4.36875, 2.12979, 0.06383],
    [1.74526, 4.34807, 1.98441, 0.06815],
    [1.61445, 4.31825, 1.80967, 0.07414],
    [1.48785, 4.27844, 1.60334, 0.08277],
    [1.36736, 4.22744, 1.41749, 0.0923],
    [1.25579, 4.1635, 1.3692, 0.09392],
    [1.15689, 4.08493, 1.3692, 0.09226],
    [1.07262, 3.99265, 1.37861, 0.09064],
    [1.00379, 3.88841, 1.4497, 0.08617],
    [0.9499, 3.77438, 1.59151, 0.07924],
    [0.90933, 3.65282, 1.74387, 0.07349],
    [0.88068, 3.52529, 1.91697, 0.06819],
    [0.86266, 3.39302, 2.14342, 0.06228],
    [0.85372, 3.25712, 2.32696, 0.05853],
    [0.85298, 3.11825, 2.5587, 0.05427],
    [0.85942, 2.97708, 2.8285, 0.04996],
    [0.87202, 2.83416, 3.0, 0.04783],
    [0.89006, 2.68988, 3.0, 0.04847],
    [0.91295, 2.54457, 3.0, 0.04903],
    [0.94043, 2.39852, 3.0, 0.04954],
    [0.97253, 2.25266, 3.0, 0.04978],
    [1.009, 2.10969, 3.0, 0.04918],
    [1.04993, 1.96995, 2.86339, 0.05085],
    [1.09551, 1.83376, 2.65352, 0.05412],
    [1.14597, 1.70145, 2.45518, 0.05767],
    [1.20179, 1.57363, 2.25731, 0.06179],
    [1.26353, 1.45096, 2.04153, 0.06727],
    [1.33187, 1.33431, 1.87485, 0.07211],
    [1.40776, 1.2249, 1.62191, 0.08209],
    [1.49204, 1.12401, 1.4497, 0.09068],
    [1.58682, 1.03457, 1.4497, 0.08989],
    [1.69383, 0.95994, 1.63196, 0.07995],
    [1.81001, 0.89771, 1.82988, 0.07202],
    [1.93328, 0.84595, 2.00739, 0.0666],
    [2.06239, 0.80345, 2.17559, 0.06248],
    [2.19642, 0.76935, 2.37408, 0.05825],
    [2.33453, 0.74275, 2.62998, 0.05348],
    [2.4759, 0.7226, 2.94855, 0.04843],
    [2.61981, 0.70783, 3.0, 0.04822],
    [2.76562, 0.69746, 3.0, 0.04873],
    [2.91287, 0.69064, 3.0, 0.04914]

]


def dist_2_points(x1, y1, x2, y2):
    return math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)


def reward_function(params):
    x = params['x']
    y = params['y']
    heading = params['heading']
    speed = params['speed']
    all_wheels_on_track = params['all_wheels_on_track']
    is_offtrack = params['is_offtrack']
    steering_angle = params['steering_angle']
    

    if is_offtrack:
        return 1e-3

    if not all_wheels_on_track:
        return 1e-3

    if steering_angle < -10:
        return 1e-3
    
    if speed < 1:
        return 1e-3
    # ======================================================================
    # 2. 找到离车最近的赛车线点
    # ======================================================================
    min_dist = 999.0
    closest_index = 0
    total_points = len(OPTIMAL_RACING_LINE)

    for i in range(total_points):
        dist = dist_2_points(x, y, OPTIMAL_RACING_LINE[i][0], OPTIMAL_RACING_LINE[i][1])
        if dist < min_dist:
            min_dist = dist
            closest_index = i

    # 获取最近点的目标速度 (第三个参数，索引为 2)
    target_speed = OPTIMAL_RACING_LINE[closest_index][2]

    # ======================================================================
    # 3. 计算各项奖励
    # ======================================================================

    # (A) 距离奖励：贴合赛车线
    MAX_ERROR = 0.15
    if min_dist > MAX_ERROR:
        distance_reward = 1e-3
    else:
        distance_reward = 1.0 - (min_dist / MAX_ERROR)

    # (B) 方向奖励：车头对准前方赛道
    # 设定一个 Lookahead 来找前方的点，保证车头平滑转向 (通常 3-4 个点比较合适)
    LOOK_AHEAD = 3
    idx_next = (closest_index + LOOK_AHEAD) % total_points
    pt_next = OPTIMAL_RACING_LINE[idx_next]

    # 计算理想航向角
    optimal_heading = math.degrees(math.atan2(pt_next[1] - y, pt_next[0] - x))
    heading_diff = abs(optimal_heading - heading)
    if heading_diff > 180:
        heading_diff = 360 - heading_diff

    # 使用余弦函数平滑惩罚角度差
    heading_reward = max(0.0, math.cos(math.radians(heading_diff)))

    # (C) 速度奖励：匹配四维数据中的目标速度
    speed_diff = abs(target_speed - speed)

    # 给予 0.3m/s 的容错区间，只要在这个区间内就算满分
    if speed_diff < 0.2:
        speed_reward = 1.0
    else:
        # 超过容错区间，差异越大分数越低
        speed_reward = max(1e-3, 1.0 - (speed_diff / 2.0))

    # ======================================================================
    # 4. 综合权重
    # ======================================================================
    # 贴线和方向最重要，速度作为辅助加成
    reward = (
            distance_reward * 0.4
            + heading_reward * 0.2
            + speed_reward * 0.4
    )

    return float(reward)


### 各步骤产出物

| Step | 产出物 | 用途 |
|------|--------|------|
| Step 1 | `waypoints.csv` | 导入 `visualize.html` 查看赛道边界 |
| Step 3 | 赛车线坐标 | 优化结果预览 |
| Step 4 | 四维数据 | 复制到 `visualize.html` 的「赛车线」文本框 |
| Step 4 | Action Space | 用于 DeepRacer 动作空间配置 |
| Step 5 | 奖励函数代码 | 直接复制到 AWS DeepRacer 控制台 |